In [5]:
import os
import glob
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import hstack

# ==============================================================================
# PHASE 1: DATA DISCOVERY AND MERGING
# ==============================================================================

# 1. Dynamically identify CSV files in the active directory
current_dir = Path.cwd()
csv_files = glob.glob("*.csv")

# Ensure the combined master file isn't accidentally included if re-running
csv_files = [f for f in csv_files if f != "master_music_df.csv"]

print(f"Current working directory: {current_dir}")
print(f"Found {len(csv_files)} CSV files to process: {csv_files}\n")

if len(csv_files) == 0:
    raise RuntimeError(
        "Stopping execution: No CSV files found. Ensure your song CSV files "
        "live in the exact same directory as this notebook."
    )

all_dfs = []

# 2. Iterate and read individual language datasets
for file in csv_files:
    try:
        df = pd.read_csv(file)
        
        # Strip tracking or accidental whitespaces from categorical keys
        for col in ['song_name', 'singer', 'language']:
            if col in df.columns:
                df[col] = df[col].astype(str).str.strip()
                
        all_dfs.append(df)
        print(f"✅ Loaded: {file} | Initial Shape: {df.shape}")
    except Exception as e:
        print(f"❌ Error processing file {file}: {e}")

# 3. Securely concatenate data layers into a unified dataset
master_music_df = pd.concat(all_dfs, ignore_index=True)

# Purge matching song duplicate instances across boundaries
master_music_df.drop_duplicates(subset=['song_name', 'singer', 'language'], inplace=True)
master_music_df.reset_index(drop=True, inplace=True)

print("\n" + "="*50)
print("MASTER DATASET COMBINED SUCCESSFULLY!")
print(f"Total Unique Tracks Across All Languages: {len(master_music_df)}")
print("="*50)
print(master_music_df['language'].value_counts())
print("="*50 + "\n")


# ==============================================================================
# PHASE 2: HYBRID RECOMMENDATION ENGINE SPECIFICATION
# ==============================================================================

class MultilingualMusicRecommender:
    def __init__(self, df):
        # Establish a hard copy to prevent SettingWithCopyWarnings during mutation
        self.df = df.copy() 
        self.similarity_matrix = None
        self.combined_features = None
        
    def prepare_features(self):
        print("Checking for missing values (NaN) across features...")
        acoustic_cols = [
            'danceability', 'acousticness', 'energy', 'liveness', 
            'loudness', 'speechiness', 'tempo', 'Valence', 'key', 'mode'
        ]
        
        # 1. Dynamic Median Imputation: Resolve NaNs column-by-column
        for col in acoustic_cols:
            nan_count = self.df[col].isna().sum()
            if nan_count > 0:
                print(f"⚠️ Found {nan_count} missing values in '{col}'. Imputing with median...")
                median_value = self.df[col].median()
                self.df[col] = self.df[col].fillna(median_value)
                
        # 2. Rescale acoustic variance to a uniform bounding range [0-1]
        scaler = MinMaxScaler()
        scaled_acoustic = scaler.fit_transform(self.df[acoustic_cols])
        
        print("Vectorizing metadata text (Song Names + Singers)...")
        # Ensure fallback text mappings for empty metadata inputs
        self.df['song_name'] = self.df['song_name'].fillna("Unknown Song")
        self.df['singer'] = self.df['singer'].fillna("Unknown Singer")
        
        # Convert internal pipe tokens to readable spaces for appropriate parsing
        self.df['metadata_text'] = self.df['song_name'] + " " + self.df['singer'].str.replace('|', ' ', regex=False)
        
        # Vectorize text representations across vocabulary dimensions
        tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
        tfidf_matrix = tfidf.fit_transform(self.df['metadata_text'])
        
        # Scale acoustic parameters slightly higher to favor structural audio similarities over text tokens
        acoustic_weight = 1.5
        weighted_acoustic = scaled_acoustic * acoustic_weight
        
        # Merge vector spaces via sparse row conversion
        self.combined_features = hstack([weighted_acoustic, tfidf_matrix]).tocsr()
        print(f"Data preparation complete! Consolidated Feature Matrix: {self.combined_features.shape}")
        
    def compute_similarity(self):
        print("Computing similarity matrix coefficients (Processing 31k+ tracks)...")
        self.similarity_matrix = cosine_similarity(self.combined_features)
        print("🎉 Recommendation engine successfully instantiated and ready for use!")
        
    def get_recommendations(self, song_title, top_n=5, allow_cross_lingual=True, target_lang=None):
        # Enforce exact string matching regardless of case variants
        idx_list = self.df[self.df['song_name'].str.lower() == song_title.lower()].index
        if len(idx_list) == 0:
            return f"Error: Song '{song_title}' could not be located in current system database."
        idx = idx_list[0]
        
        # Extract dot product scores and sort by decreasing metric distance
        sim_scores = list(enumerate(self.similarity_matrix[idx]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        recommendations = []
        source_lang = self.df.iloc[idx]['language']
        
        for i, score in sim_scores:
            if i == idx:
                continue  # Exclude target search query song item itself
                
            candidate_lang = self.df.iloc[i]['language']
            
            # Apply dynamic routing constraints
            if not allow_cross_lingual and candidate_lang != source_lang:
                continue
            if target_lang and candidate_lang.lower() != target_lang.lower():
                continue
                
            recommendations.append({
                'Song Name': self.df.iloc[i]['song_name'],
                'Singer(s)': self.df.iloc[i]['singer'],
                'Language': candidate_lang,
                'Vibe Match Score': round(float(score), 4)
            })
            
            if len(recommendations) == top_n:
                break
                
        return pd.DataFrame(recommendations)


# ==============================================================================
# PHASE 3: ENGINE INITIALIZATION & VERIFICATION QUERIES
# ==============================================================================

# Instantiate engine profile using the compiled dataframe
engine = MultilingualMusicRecommender(master_music_df)
engine.prepare_features()
engine.compute_similarity()

print("\n" + "="*50)
print("RUNNING PIPELINE VERIFICATION TEST CASES")
print("="*50)

# Scenario A: Validate Cross-Lingual Audio Signature Mapping
print("\n[Test A: Cross-Lingual Discovery for 'Pehle Lalkare Naal']")
res_a = engine.get_recommendations(song_title="Pehle Lalkare Naal", top_n=5, allow_cross_lingual=True)
print(res_a.to_string(index=False))

# Scenario B: Validate Native Language-Locked Boundary Mapping
print("\n[Test B: Native Language-Locked Filter for 'En Iniya Pon Nilave']")
res_b = engine.get_recommendations(song_title="En Iniya Pon Nilave", top_n=5, allow_cross_lingual=False)
print(res_b.to_string(index=False))

# Scenario C: Validate Targeted Translation Mapping
print("\n[Test C: Target Domain Translation - 'Manchu Kurise' mapped to Gujarati]")
res_c = engine.get_recommendations(song_title="Manchu Kurise", top_n=5, target_lang="Gujarati")
print(res_c.to_string(index=False))

Current working directory: c:\Users\asmit\Downloads\songs recommendation system
Found 16 CSV files to process: ['Assamese_songs.csv', 'Bengali_songs.csv', 'Bhojpuri_songs.csv', 'Gujarati_songs.csv', 'Haryanvi_songs.csv', 'Hindi_songs.csv', 'Kannada_songs.csv', 'Malayalam_songs.csv', 'Marathi_songs.csv', 'Odia_songs.csv', 'Old_songs.csv', 'Punjabi_songs.csv', 'Rajasthani_songs.csv', 'Tamil_songs.csv', 'Telugu_songs.csv', 'Urdu_songs.csv']

✅ Loaded: Assamese_songs.csv | Initial Shape: (724, 19)
✅ Loaded: Bengali_songs.csv | Initial Shape: (958, 19)
✅ Loaded: Bhojpuri_songs.csv | Initial Shape: (519, 19)
✅ Loaded: Gujarati_songs.csv | Initial Shape: (2115, 19)
✅ Loaded: Haryanvi_songs.csv | Initial Shape: (228, 19)
✅ Loaded: Hindi_songs.csv | Initial Shape: (1184, 19)
✅ Loaded: Kannada_songs.csv | Initial Shape: (3559, 19)
✅ Loaded: Malayalam_songs.csv | Initial Shape: (479, 19)
✅ Loaded: Marathi_songs.csv | Initial Shape: (4699, 19)
✅ Loaded: Odia_songs.csv | Initial Shape: (940, 19)
✅ 